# Mapping Climate Crises with Python
### MLH Global Hack Week — Solution Notebook

In this workshop we'll take the **EM-DAT International Disaster Database** — a real, messy, 16,000+ row dataset — and turn it into an interactive map of climate-related natural disasters.

**Plan:**
1. Load and explore the raw data
2. Clean and filter it down to what we need
3. Explore patterns with pandas
4. Build an interactive map with Folium
5. Mini challenge — make it your own!


## 0. Setup

Get the dataset yourself from [public.emdat.be/data](https://public.emdat.be/data) (free account required). Filter to 2000–2026, leave Classification/Countries blank, leave historical toggle off. Download as `.xlsx` and place it in the `data/` folder.

In [ ]:
!pip install pandas folium openpyxl -q

In [ ]:
import pandas as pd
import folium

df = pd.read_excel('../data/public_emdat_2026-05-25.xlsx')
print(df.shape)
df.head()

## 1. Explore the mess

47 columns, 16,834 rows. Let's see what we're working with.

In [ ]:
df.info()

In [ ]:
# How many disaster groups do we have?
df['Disaster Group'].value_counts()

We've got two groups: **Natural** disasters (floods, earthquakes, storms...) and **Technological** (industrial accidents, transport accidents). 

For "climate crises" we want the Natural ones.

In [ ]:
# Filter to natural disasters only
natural = df[df['Disaster Group'] == 'Natural'].copy()
print(f"Natural disasters: {len(natural)} rows")
natural['Disaster Type'].value_counts()

## 2. The lat/long problem

To put events on a map we need coordinates. Let's check how many rows actually have them.

In [ ]:
print(f"Rows with coordinates: {natural['Latitude'].notna().sum()} / {len(natural)}")
print(f"That's {natural['Latitude'].notna().sum() / len(natural) * 100:.1f}% of natural disasters")

Only about 17% of rows have coordinates. This is a **real data quality issue** — it means our map will only show a *subset* of disasters, and that subset might not be representative (e.g. maybe certain disaster types or regions are geocoded more often than others).

For this workshop we'll proceed with what we have, but it's worth flagging: **any conclusions from the map come with a caveat about this missing data.**

In [ ]:
# Keep only rows with coordinates
geo = natural.dropna(subset=['Latitude', 'Longitude']).copy()
print(f"Remaining rows: {len(geo)}")
geo['Disaster Type'].value_counts()

## 3. Select and clean the columns we need

We don't need all 47 columns. Let's pick the ones relevant to mapping and severity.

In [ ]:
cols = ['Country', 'Disaster Type', 'Start Year', 'Latitude', 'Longitude', 'Total Deaths', 'Total Affected']
geo = geo[cols]
geo.head()

In [ ]:
# Check for nulls in our remaining columns
geo.isna().sum()

`Total Deaths` and `Total Affected` have some missing values. For mapping purposes (marker size), we'll treat missing as 0 — but in a real analysis you'd want to think carefully about whether "missing" really means "zero" or just "not recorded".

In [ ]:
geo['Total Deaths'] = geo['Total Deaths'].fillna(0)
geo['Total Affected'] = geo['Total Affected'].fillna(0)
geo = geo.dropna(subset=['Country', 'Start Year'])  # drop anything still missing essentials
geo.head()

## 4. Quick exploration with pandas

Now that it's clean, let's ask some questions of the data.

In [ ]:
# Which disaster types are most common in our geocoded data?
geo['Disaster Type'].value_counts()

In [ ]:
# Which countries have the most recorded events?
geo['Country'].value_counts().head(10)

In [ ]:
# Filter to a specific time range, e.g. 2020 onwards
recent = geo[geo['Start Year'] >= 2020]
print(f"Events since 2020: {len(recent)}")
recent['Disaster Type'].value_counts()

## 5. Building the map with Folium

Let's start simple — a blank world map.

In [ ]:
m = folium.Map(location=[20, 0], zoom_start=2)
m

Now let's add a single marker to check the syntax works.

In [ ]:
m = folium.Map(location=[20, 0], zoom_start=2)

folium.Marker(
    location=[geo.iloc[0]['Latitude'], geo.iloc[0]['Longitude']],
    popup=f"{geo.iloc[0]['Disaster Type']} in {geo.iloc[0]['Country']} ({geo.iloc[0]['Start Year']})"
).add_to(m)

m

## 6. Plotting every event

Now let's loop through the dataframe and add a marker for every disaster. We'll:
- Use `CircleMarker` instead of `Marker` (lighter weight for many points)
- Colour-code by disaster type
- Size by `Total Affected` (so bigger circles = bigger impact)
- Add a popup with details

In [ ]:
# Assign a colour to each disaster type
disaster_types = geo['Disaster Type'].unique()
colours = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'cadetblue', 'darkgreen', 'pink', 'gray', 'black', 'lightblue', 'darkpurple', 'beige']
colour_map = dict(zip(disaster_types, colours))
colour_map

In [ ]:
import math

m = folium.Map(location=[20, 0], zoom_start=2)

for _, row in geo.iterrows():
    # scale radius by Total Affected, with a sensible floor and cap
    affected = row['Total Affected']
    radius = 3 + math.log10(affected + 1) * 2  # log scale so huge numbers don't dominate

    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=radius,
        color=colour_map.get(row['Disaster Type'], 'gray'),
        fill=True,
        fill_opacity=0.6,
        popup=folium.Popup(
            f"<b>{row['Disaster Type']}</b><br>"
            f"{row['Country']}, {int(row['Start Year'])}<br>"
            f"Deaths: {int(row['Total Deaths'])}<br>"
            f"Affected: {int(row['Total Affected']):,}",
            max_width=250
        )
    ).add_to(m)

m

## 7. Mini Challenge (10 minutes)

Pick one (or more!) of these to try:

1. **Filter to your region** — e.g. `geo[geo['Country'].str.contains('United Kingdom')]` and replot
2. **Filter to one disaster type** — e.g. just floods or just earthquakes, and see how the pattern changes
3. **Change the colour scheme** — maybe colour by *decade* instead of disaster type
4. **Add a legend** — Folium supports custom HTML legends, can you add one showing what each colour means?
5. **Save your map** — `m.save('my_climate_map.html')` and open it in a browser

Use the code cell below to experiment!

In [ ]:
# Your code here!


## 8. Wrap-up

You've taken a real, messy 16,000-row government dataset and turned it into an interactive map — handling missing data, filtering, and visualisation along the way.

**Where to go next:**
- Try the [GDIS dataset](https://www.nature.com/articles/s41597-021-00846-6) for subnational geocoding
- Explore `folium.plugins` — heatmaps, marker clusters, time-based animations (`TimestampedGeoJson`)
- Combine with [Our World in Data](https://ourworldindata.org/) climate datasets for richer context

Code for this workshop: **https://github.com/TalenMud/GHW-Workshop-Mapping-Climate-Crises-with-Python**

Thanks for joining!